# Data Preparation

## Konfiguration

In [2]:
# @title: "Setup"

from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/JuliusD9500/modern-fair-credit-scoring.git" #GitHub Repo URL
BRANCH = "main" #Git branch to clone
REPO_DIR = Path("/content/modern-fair-credit-scoring") #Local Colab directory to clone the repo into

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)

print(f"Repository bereit: {REPO_DIR}")

Repository bereit: /content/modern-fair-credit-scoring


## Read local data

In [9]:
# @title: "Einlesen lokaler Daten"
import pandas as pd

DATA_PATH = (
    REPO_DIR
    / "data"
    / "raw"
    / "south_german_credit"
    / "SouthGermanCredit.asc"
)

dat = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=0,
)

nam_fahrmeirbook = [
    "laufkont", "laufzeit", "moral", "verw",
    "hoehe", "sparkont", "beszeit", "rate",
    "famges", "buerge", "wohnzeit", "verm",
    "alter", "weitkred", "wohn", "bishkred",
    "beruf", "pers", "telef", "gastarb",
    "kredit",
]

nam_evtree = [
    "status", "duration", "credit_history",
    "purpose", "amount", "savings", "employment_duration",
    "installment_rate", "personal_status_sex",
    "other_debtors", "present_residence", "property",
    "age", "other_installment_plans", "housing",
    "number_credits", "job", "people_liable", "telephone",
    "foreign_worker", "credit_risk",
]

dat.columns = nam_evtree

dat.head()

,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,1,18,4,2,1049,1,2,4,2,1,...,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,...,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,...,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,...,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,...,2,38,1,2,2,2,2,1,1,1


In [10]:
# Alle Variablen außer duration, purpose, amount und age werden Faktoren.
excluded = {"duration", "purpose", "amount", "age"}

for column in dat.columns:
    if column not in excluded:
        dat[column] = pd.Categorical(dat[column].astype(str))

# purpose wird separat Faktor; auch die unbeobachtete Stufe 8 bleibt enthalten.
dat["purpose"] = pd.Categorical(
    dat["purpose"].astype(str),
    categories=[str(value) for value in range(11)],
)

level_labels = {
    "credit_risk": [
        "bad",
        "good",
    ],
    "status": [
        "no checking account",
        "... < 0 DM",
        "0<= ... < 200 DM",
        "... >= 200 DM / salary for at least 1 year",
    ],
    "credit_history": [
        "delay in paying off in the past",
        "critical account/other credits elsewhere",
        "no credits taken/all credits paid back duly",
        "existing credits paid back duly till now",
        "all credits at this bank paid back duly",
    ],
    "purpose": [
        "others",
        "car (new)",
        "car (used)",
        "furniture/equipment",
        "radio/television",
        "domestic appliances",
        "repairs",
        "education",
        "vacation",
        "retraining",
        "business",
    ],
    "savings": [
        "unknown/no savings account",
        "... < 100 DM",
        "100 <= ... < 500 DM",
        "500 <= ... < 1000 DM",
        "... >= 1000 DM",
    ],
    "employment_duration": [
        "unemployed",
        "< 1 yr",
        "1 <= ... < 4 yrs",
        "4 <= ... < 7 yrs",
        ">= 7 yrs",
    ],
    "other_debtors": [
        "none",
        "co-applicant",
        "guarantor",
    ],
    "personal_status_sex": [
        "male : divorced/separated",
        "female : non-single or male : single",
        "male : married/widowed",
        "female : single",
    ],
    "property": [
        "unknown / no property",
        "car or other",
        "building soc. savings agr. / life insurance",
        "real estate",
    ],
    "other_installment_plans": [
        "bank",
        "stores",
        "none",
    ],
    "housing": [
        "for free",
        "rent",
        "own",
    ],
    "job": [
        "unemployed/unskilled - non-resident",
        "unskilled - resident",
        "skilled employee/official",
        "manager/self-empl/highly qualif. employee",
    ],
    "people_liable": [
        "3 or more",
        "0 to 2",
    ],
    "telephone": [
        "no",
        "yes (under customer name)",
    ],
    "foreign_worker": [
        "yes",
        "no",
    ],
}

for column, labels in level_labels.items():
    dat[column] = dat[column].cat.rename_categories(labels)

## Export

In [11]:
import csv

OUTPUT_DIR = REPO_DIR / "data" / "prepared"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dat.to_csv(
    OUTPUT_DIR / "german.data",
    sep=" ",
    index=False,
    quoting=csv.QUOTE_NONE,
    escapechar="\\",
)

dat.to_pickle(OUTPUT_DIR / "german.pkl")

dat.to_csv(
    OUTPUT_DIR / "german.csv",
    index=False,
    quoting=csv.QUOTE_NONE,
    escapechar="\\",
)